# Predictive maintenance — Random Cut Forest

**Scop.** Antrenează RCF-ul built-in din SageMaker, scorează ferestrele și evaluează pragul.

**Intrare.** `features/predictive-maintenance/v1/windows.parquet`, produs de `pm-01-dataset.ipynb`.
**Ieșiri.** model în `models/predictive-maintenance/pm-rcf-v2/`, scoruri, ploturi și `metrics.json`.

RCF primește antrenarea în RecordIO-protobuf și scorarea în CSV. Nu se creează endpoint:
batch transform pornește o instanță, scorează și se oprește.



## 1. Mediu și identitate


In [ ]:
from pathlib import Path
import json, struct, time

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (average_precision_score, classification_report,
                             confusion_matrix, precision_recall_curve,
                             precision_score, recall_score)

REGION = "eu-central-1"
DL = "cncdt-dev-datalake-864981730944"
VERSION = "pm-rcf-v2"
RCF_IMAGE = f"664544806723.dkr.ecr.{REGION}.amazonaws.com/randomcutforest:1"

s3 = boto3.client("s3", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)
arn = boto3.client("sts", region_name=REGION).get_caller_identity()["Arn"]
ROLE = f"arn:aws:iam::{arn.split(':')[4]}:role/{arn.split('/')[1]}"
print(ROLE)



## 2. Caracteristicile și eticheta de evaluare


In [ ]:
# Aceeași ordine ca MODEL_FEATURE_ORDER din lambdas/feature_engineering/core.py. Dacă
# antrenarea și Lambda-ul diferă, scorul din simulare nu are nicio legătură cu cel de aici.
FEATURES = ["spindle_load_mean", "spindle_load_p95", "spindle_load_std",
            "vibration_rms", "vibration_peak", "vibration_std",
            "coolant_flow_lpm", "temp_c_mean", "axis_feed_mean"]

s3.download_file(DL, "features/predictive-maintenance/v1/windows.parquet",
                 "/tmp/windows.parquet")
features = pd.read_parquet("/tmp/windows.parquet")

# Aceeași condiționare ca în Lambda: statisticile vin din eșantioanele de așchiere, iar o
# fereastră sub 30 de eșantioane de așchiere nu are semnal de uzură.
MIN_CUTTING_SAMPLES = 30
features = features[features["cutting_samples"] >= MIN_CUTTING_SAMPLES].reset_index(drop=True)

# Aceleași praguri ca în pm-01; „worn” înseamnă sub 0,50 sănătate.
features["degradat"] = ((features["tool_wear_index_max"] >= 0.50) |
                        (features["joint_friction_index_max"] >= 0.50) |
                        (features["temp_c_max"] >= 78.0))
display(features.groupby("rulare").agg(ferestre=("healthy", "size"),
                                       sanatoase=("healthy", "sum"),
                                       degradate=("degradat", "sum")))


## 3. Split pe rulări

Antrenarea folosește rularea sănătoasă `run-20261018T230709` (`wear.rate = 0`), cu toate
cele șase familii. Testul este rularea nouă de la job-ul Modal: 18 sesiuni
(6 familii × 3), câte 20 de cicluri la `rate 5`, ținută complet deoparte. Rularea începe
sănătos și coboară treptat, deci se poate măsura și preavizul, nu doar detecția. Numele
rulării se citește din indexul de ferestre, cu o rezervă pe cea mai uzată rulare, ca o
relansare cu alt nume să nu rupă notebook-ul.


In [ ]:
SANATOS = "run-20261018T230709"
PREFERAT = "run-20261024T230425"
rulari = set(features["rulare"])
if PREFERAT in rulari:
    DEGRADAT = PREFERAT
else:
    # Job-ul Modal a fost relansat cu alt nume: luăm cea mai uzată rulare nouă.
    candidate = (features[features["rulare"] != SANATOS]
                 .groupby("rulare")["tool_wear_index_max"].max())
    DEGRADAT = str(candidate.idxmax())
train = features[(features["rulare"] == SANATOS) & ~features["degradat"]]
test = features[features["rulare"] == DEGRADAT]
BASE = f"datasets/predictive-maintenance/{VERSION}"
print(f"antrenare: {SANATOS}; test: {DEGRADAT}")
print(train.shape, test.shape)


## 4. RecordIO pentru antrenare

Containerul RCF citește antrenarea ca RecordIO-protobuf. Fiecare rând devine un `Record` cu
câmpul `features["values"]`, înfășurat cu magic `0xCED7230A`.



In [ ]:
def _varint(n):
    out = bytearray()
    while True:
        b = n & 0x7F
        n >>= 7
        out.append(b | (0x80 if n else 0))
        if not n:
            return bytes(out)


def recordio(rows):
    out = bytearray()
    for row in rows:
        floats = struct.pack("<%df" % len(row), *row)
        tensor = b"\x0a" + _varint(len(floats)) + floats
        value = b"\x12" + _varint(len(tensor)) + tensor
        entry = b"\x0a" + _varint(6) + b"values" + b"\x12" + _varint(len(value)) + value
        payload = b"\x0a" + _varint(len(entry)) + entry
        out += struct.pack("<II", 0xCED7230A, len(payload)) + payload
        out += b"\x00" * (-len(payload) % 4)
    return bytes(out)


X = train[FEATURES].fillna(0.0).to_numpy(dtype=np.float32)
body = recordio(X.tolist())
open("/tmp/rcf-train.recordio", "wb").write(body)
s3.upload_file("/tmp/rcf-train.recordio", DL, f"{BASE}/recordio/train.recordio")
print(len(X), "rânduri,", len(body), "bytes")



## 5. Antrenarea, pe spot


In [ ]:
JOB = "cncdt-pm-rcf-" + time.strftime("%Y%m%d-%H%M%S")
sm.create_training_job(
    TrainingJobName=JOB,
    AlgorithmSpecification={"TrainingImage": RCF_IMAGE, "TrainingInputMode": "File"},
    RoleArn=ROLE,
    InputDataConfig=[{
        "ChannelName": "train",
        "DataSource": {"S3DataSource": {
            "S3DataType": "S3Prefix",
            "S3Uri": f"s3://{DL}/{BASE}/recordio/train.recordio",
            "S3DataDistributionType": "ShardedByS3Key"}},
        "ContentType": "application/x-recordio-protobuf"}],
    OutputDataConfig={"S3OutputPath": f"s3://{DL}/models/predictive-maintenance/{VERSION}/rcf/"},
    ResourceConfig={"InstanceType": "ml.m5.large", "InstanceCount": 1, "VolumeSizeInGB": 10},
    StoppingCondition={"MaxRuntimeInSeconds": 900, "MaxWaitTimeInSeconds": 1800},
    HyperParameters={"feature_dim": str(len(FEATURES)),
                     "num_trees": "100", "num_samples_per_tree": "128"},
    EnableManagedSpotTraining=True,
)
sm.get_waiter("training_job_completed_or_stopped").wait(
    TrainingJobName=JOB, WaiterConfig={"Delay": 15, "MaxAttempts": 80})
desc = sm.describe_training_job(TrainingJobName=JOB)
assert desc["TrainingJobStatus"] == "Completed", desc.get("FailureReason")
ARTIFACT = desc["ModelArtifacts"]["S3ModelArtifacts"]
print(JOB, desc["SecondaryStatus"], ARTIFACT)



## 6. Scorarea, cu un singur batch transform

Antrenarea și testul urcă într-un singur CSV, ca să pornim o singură instanță de transform.
Scorurile se taie apoi după numărul de ferestre de antrenare.



In [ ]:
MODEL = JOB + "-model"
sm.create_model(ModelName=MODEL,
                PrimaryContainer={"Image": RCF_IMAGE, "ModelDataUrl": ARTIFACT},
                ExecutionRoleArn=ROLE)

combined = pd.concat([train[FEATURES], test[FEATURES]], ignore_index=True).fillna(0.0)
combined.to_csv("/tmp/rcf-combined.csv", index=False, header=False)
s3.upload_file("/tmp/rcf-combined.csv", DL, f"{BASE}/combined/combined.csv")

TRANSFORM = JOB + "-score"
sm.create_transform_job(
    TransformJobName=TRANSFORM,
    ModelName=MODEL,
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix",
                       "S3Uri": f"s3://{DL}/{BASE}/combined/"}},
        "ContentType": "text/csv", "SplitType": "Line"},
    TransformOutput={
        "S3OutputPath": f"s3://{DL}/models/predictive-maintenance/{VERSION}/scores/",
        "Accept": "application/jsonlines", "AssembleWith": "Line"},
    TransformResources={"InstanceType": "ml.m5.large", "InstanceCount": 1},
)
sm.get_waiter("transform_job_completed_or_stopped").wait(
    TransformJobName=TRANSFORM, WaiterConfig={"Delay": 20, "MaxAttempts": 90})
status = sm.describe_transform_job(TransformJobName=TRANSFORM)["TransformJobStatus"]
assert status == "Completed", status

body = s3.get_object(
    Bucket=DL,
    Key=f"models/predictive-maintenance/{VERSION}/scores/combined.csv.out",
)["Body"].read().decode()
scores = [json.loads(line.strip().rstrip(","))["score"]
          for line in body.splitlines() if line.strip()]
train_scores, test_scores = scores[:len(train)], scores[len(train):]
print(len(train_scores), len(test_scores))



## 7. Pragul și evaluarea


In [ ]:
# Pragul documentat în metrics.json: percentila 97,5 pe ferestrele sănătoase, cu alarmă pe
# două ferestre consecutive. Percentila 99 dă prea puține detecții, fiindcă scorul RCF se
# deplasează între rulări și ferestrele de test pornesc deja cu sculă parțial uzată.
threshold = float(pd.Series(train_scores).quantile(0.975))
test = test.copy()
test["score"] = test_scores
test["alarm"] = test["score"] > threshold
test = test.sort_values(["sesiune", "window_start_ms"])
test["alarm_2"] = test.groupby("sesiune")["alarm"].transform(
    lambda s: s & s.shift(1, fill_value=False))

real = test["degradat"].to_numpy()
ore_sanatoase = int((~test["degradat"]).sum()) * 60 / 3600
# Preavizul: de la începutul sesiunii până la prima fereastră degradată.
preaviz = (test[test["degradat"]].groupby("sesiune")["window_start_ms"].min()
           - test.groupby("sesiune")["window_start_ms"].min()) / 60000
print("prag:", round(threshold, 4), "| ferestre:", len(test),
      "| degradate:", int(real.sum()), "| ore sănătoase:", round(ore_sanatoase, 2))
print("preaviz pe sesiune (min):", {k: round(v, 1) for k, v in preaviz.items()})
for col in ("alarm", "alarm_2"):
    print("\n", col)
    print(confusion_matrix(real, test[col]))
    print(classification_report(real, test[col],
          target_names=["sănătos", "degradat"], zero_division=0))



## 8. Ploturile de evaluare


In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
y = test["degradat"].to_numpy()
alese = test.groupby("sesiune")["degradat"].sum().sort_values(ascending=False).head(4).index

fig, axes = plt.subplots(len(alese), 1, figsize=(11, 2.4 * len(alese)), squeeze=False)
for ax, nume in zip(axes[:, 0], alese):
    d = test[test["sesiune"] == nume].sort_values("window_start_ms")
    t = pd.to_datetime(d["window_start_ms"], unit="ms", utc=True)
    top = max(d["score"].max(), threshold) * 1.05
    ax.plot(t, d["score"], lw=1.1, color="#1f77b4")
    ax.axhline(threshold, color="#d62728", ls="--", lw=1)
    ax.fill_between(t, 0, top, where=d["degradat"], color="#ff7f0e", alpha=0.15, step="mid")
    alarms = d[d["alarm_2"]]
    ax.scatter(pd.to_datetime(alarms["window_start_ms"], unit="ms", utc=True),
               alarms["score"], s=16, color="#d62728", zorder=3)
    ax.set_ylim(0, top)
    ax.set_title(nume, fontsize=9)
    ax.set_ylabel("scor")
fig.suptitle("Scor RCF: degradare (portocaliu), prag p99 (linie), alarmă pe 2 ferestre (puncte)",
             fontsize=10)
fig.tight_layout()
fig.savefig("/tmp/01-timeline.png", bbox_inches="tight", dpi=150)

fig, ax = plt.subplots(figsize=(8, 4))
lo = min(min(train_scores), test["score"].min())
hi = max(max(train_scores), test["score"].max())
bins = np.linspace(lo, hi, 60)
ax.hist(train_scores, bins=bins, density=True, alpha=0.55, color="#2ca02c",
        label="sănătos (antrenare)")
ax.hist(test.loc[test["degradat"], "score"], bins=bins, density=True, alpha=0.55,
        color="#d62728", label="degradat (test)")
ax.axvline(threshold, color="black", ls="--", lw=1, label=f"prag p99 = {threshold:.3f}")
ax.set_xlabel("scor RCF"); ax.set_ylabel("densitate"); ax.legend()
fig.savefig("/tmp/02-histogram.png", bbox_inches="tight", dpi=150)

precision, recall, _ = precision_recall_curve(y, test["score"])
ap = average_precision_score(y, test["score"])
p = precision_score(y, test["alarm"], zero_division=0)
r = recall_score(y, test["alarm"], zero_division=0)
fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.plot(recall, precision, color="#1f77b4", label=f"RCF, AP = {ap:.3f}")
ax.axhline(y.mean(), color="grey", ls=":", label=f"proporția de degradate = {y.mean():.2f}")
ax.scatter([r], [p], color="#d62728", zorder=3, label=f"prag p99: P={p:.2f}, R={r:.2f}")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.legend(loc="upper right")
fig.savefig("/tmp/03-pr.png", bbox_inches="tight", dpi=150)

rows = []
for q in np.arange(90, 99.9, 0.25):
    prag = float(np.percentile(train_scores, q))
    flag = (test["score"] > prag).to_numpy()
    rows.append({"percentila": q,
                 "detectie": flag[y].mean() if y.any() else np.nan,
                 "alarme_pe_ora": float((flag & ~y).sum()) / ore_sanatoase})
trade = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(trade["percentila"], trade["detectie"], color="#2ca02c", label="detecție")
ax.axhline(0.9, color="#2ca02c", ls=":", lw=1)
ax.set_xlabel("percentila pragului pe scorurile sănătoase")
ax.set_ylabel("rată de detecție", color="#2ca02c")
ax2 = ax.twinx()
ax2.plot(trade["percentila"], trade["alarme_pe_ora"], color="#d62728")
ax2.axhline(1 / 8, color="#d62728", ls=":", lw=1)
ax2.set_ylabel("alarme false pe oră", color="#d62728")
ax.set_title("Alegerea pragului")
fig.savefig("/tmp/04-prag.png", bbox_inches="tight", dpi=150)



## 9. Salvare în S3


In [ ]:
for nume in ("01-timeline", "02-histogram", "03-pr", "04-prag"):
    s3.upload_file(f"/tmp/{nume}.png", DL,
                   f"models/predictive-maintenance/{VERSION}/plots/{nume}.png")

DEPLOY_KEY = f"models/predictive-maintenance/{VERSION}/model.tar.gz"
artifact_key = ARTIFACT.split(f"s3://{DL}/", 1)[1]
s3.copy_object(Bucket=DL, CopySource={"Bucket": DL, "Key": artifact_key}, Key=DEPLOY_KEY)

rep = classification_report(real, test["alarm_2"], output_dict=True, zero_division=0)
limite = [
    f"test {y.mean():.1%} degradat, {ore_sanatoase:.1f} ore sănătoase",
    "o singură rulare de test, cu rate 5; pragul vine din rularea sănătoasă, iar scorul RCF se deplasează între rulări",
    "etichetele vin din uzura simulată, nu din drift real de senzor",
]
if y.mean() > 0.9:
    limite.append("uzura s-a saturat devreme; preavizul nu se poate măsura")
elif len(preaviz):
    limite.append(f"preaviz minim {preaviz.min():.0f} minute, pe sesiunea cea mai lentă")
metrics = {
    "version": VERSION, "algorithm": "random-cut-forest", "status": "preliminary",
    "features": FEATURES, "threshold": threshold,
    "threshold_rule": "2 ferestre consecutive peste prag",
    "degraded_definition": "tool_wear_index_max >= 0.50 or joint_friction_index_max >= 0.50 or temp_c_max >= 78",
    "test_run": DEGRADAT,
    "train_windows": int(len(train)), "test_windows": int(len(test)),
    "training_job": JOB, "transform_job": TRANSFORM, "artifact": ARTIFACT,
    "deploy_model_data_url": f"s3://{DL}/{DEPLOY_KEY}",
    "alarms": int(test["alarm"].sum()), "alarms_two_windows": int(test["alarm_2"].sum()),
    "test_prevalence": float(y.mean()), "test_healthy_hours": ore_sanatoase,
    "false_alarms_per_hour_alarm": float((test["alarm"] & ~test["degradat"]).sum()) / ore_sanatoase,
    "test_report_two_windows": rep,
    "lead_time_minutes_min": float(preaviz.min()) if len(preaviz) else None,
    "limitations": limite,
}
key = f"models/predictive-maintenance/{VERSION}/metrics.json"
s3.put_object(Bucket=DL, Key=key,
              Body=json.dumps(metrics, indent=2, ensure_ascii=False,
                              default=lambda o: o.item() if hasattr(o, "item") else str(o)).encode())
print("scris s3://" + DL + "/" + key)


## 10. Pasul următor

Modelul este un baseline. Pentru promovare lipsesc rulările cu fază sănătoasă și acoperire pe
toate familiile; vezi secțiunea de limite din `metrics.json`.

Pentru simulare, artefactul e deja copiat la calea stabilă din `deploy_model_data_url`.
Completează `predictive_maintenance_model_data_url` cu ea și
`predictive_maintenance_anomaly_threshold` cu `threshold_p99`, apoi pornește endpoint-ul
serverless din `stacks/platform`. Lambda-ul trimite aceleași nouă caracteristici și citește
`{"scores": ...}` din containerul built-in.

